# Aralin 13 - Memorya ng Ahente


## Setup

Ipinapakita ng notebook na ito kung paano gumawa ng travel booking agent na may **persistent memory** gamit ang **Microsoft Agent Framework** (MAF).

Malalaman mo kung paano nakakaapekto ang iba't ibang uri ng agent memory — working, short-term, at long-term — sa kung paano pinananatili at ginagamit ng agent ang impormasyon sa mga pag-uusap.

**Mga Kinakailangan:**
- Isang Azure AI Foundry project na may deployed chat model (hal. `gpt-4o-mini`).
- Nakalog-in gamit ang Azure CLI — patakbuhin ang `az login` sa iyong terminal.
- `AZURE_AI_PROJECT_ENDPOINT` — ang endpoint ng iyong Azure AI Foundry project.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ang pangalan ng iyong deployed model.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Mga Uri ng Memorya ng Ahente

Maaaring gamitin ng mga AI agent ang iba't ibang uri ng memorya, na bawat isa ay may kanya-kanyang layunin:

### Working Memory
Ang mismong usapan — ang mga mensaheng ipinagpalitan sa isang session. Maaaring balikan ng ahente ang mga naunang mensahe sa parehong usapan upang mapanatili ang pagkakaugnay-ugnay. Sa MAF, ito ay ginagawa gamit ang **`agent.create_session()`**, na nagbabalik ng `AgentSession`.

### Short-Term Memory
Impormasyon na nananatili habang tumatagal ang isang gawain o session ngunit hindi permanenteng iniimbak. Halimbawa, maaaring mangalap ang ahente ng mga katotohanan sa panahon ng multi-turn planning na pag-uusap at gamitin ang mga ito upang makabuo ng huling itineraryo.

### Long-Term Memory
Mga kagustuhan at katotohanan na nananatili **sa buong mga session**. Hindi na kailangang ulitin ng bumabalik na user ang kanilang mga dietary restrictions o estilo ng paglalakbay. Kadalasang sinusuportahan ang long-term memory ng isang panlabas na imbakan — isang database, file, o vector index — at inilalabas sa ahente sa pamamagitan ng mga tool.


## Working Memory with Sessions

Ang pinakasimpleng anyo ng memorya ay ang conversation session. Kapag ipinasa mo ang parehong session (na nilikha gamit ang `agent.create_session()`) sa magkakasunod na `agent.run()` na tawag, nakikita ng agent ang buong kasaysayan ng pag-uusap na iyon at maaaring maalala ang mga naunang detalye.

Gumawa tayo ng travel agent at ipakita ang working memory.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Tamang na naalala ng ahente ang badyet dahil pareho ang sesyon ng dalawang mensahe. Ito ay **working memory** — umiiral lamang ito sa habang buhay ng sesyon.

### Ano ang nangyayari sa bagong thread?

Kung gagawa tayo ng **bagong** sesyon, wala nang alaala ang ahente tungkol sa nakaraang usapan:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Long-Term Memory Pattern

Upang maalala ang mga preference ng user **sa iba't ibang session**, kailangan natin ng isang persistenteng imbakan na nabubuhay sa labas ng thread ng pag-uusap. Ina-access ng ahente ang imbakan na ito sa pamamagitan ng **mga tool** — mga function na maaari nitong tawagan upang mag-imbak at kumuha ng impormasyon.

Sa ibaba, nag-implement tayo ng isang simpleng in-memory preference store (sa produksyon, susuportahan mo ito ng isang database o vector index) at ipinapakita ito bilang mga tool na magagamit ng ahente.

### Architecture
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenario 1 — Unang beses na gumagamit ay nagbu-book ng isang anibersaryo na biyahe

Si Sarah ay bumisita sa unang pagkakataon. Dapat itala ng ahente ang kanyang mga kagustuhan gamit ang mga kagamitan at gamitin ito upang magrekomenda ng mga hotel.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Scenario 2 — Bumalik si Sarah makalipas ang ilang linggo

Nagsimula si Sarah ng isang **bagong thread** (nagsusimulate ng bagong sesyon). Walang laman ang working memory, ngunit nasa preference store pa rin ang kanyang impormasyon. Dapat itong kunin ng agent at gamitin upang personalisahin ang mga rekomendasyon.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Buod

Sa araling ito natutunan mo ang tatlong uri ng memorya ng ahente at kung paano ito ipatupad gamit ang Microsoft Agent Framework:

| Uri ng Memorya | Mekanismo ng MAF | Habang-Buhay |
|---|---|---|
| **Working** | `agent.create_session()` | Isang pag-uusap lang |
| **Pangmadaliang panandalian** | Naipong konteksto sa loob ng isang thread | Isang gawain / sesyon lang |
| **Pangmatagalan** | Panlabas na imbakan na nae-access sa pamamagitan ng `@tool` functions | Sa pagitan ng mga sesyon |

### Pangunahing Punto
1. **`agent.create_session()`** ay nagbibigay ng working memory — nakikita ng ahente ang buong kasaysayan ng pag-uusap sa loob ng isang sesyon.
2. **Ang mga bagong sesyon ay nawawala ang konteksto** — kung walang pangmatagalang memorya, hindi matandaan ng ahente ang mga nakaraang pag-uusap.
3. **Ang mga `@tool` functions** ang nagpapatulay — pinapayagan nila ang ahente na mag-imbak at kumuha ng impormasyon mula sa isang permanenteng imbakan.
4. **Lumalago ang personalisasyon sa paglipas ng panahon** — habang mas maraming kagustuhan ang naitatago, mas gumaganda ang mga rekomendasyon ng ahente.

### Mga Totoong Aplikasyon
- **Serbisyo sa Customer**: Tandaan ang kasaysayan at mga kagustuhan ng customer
- **Personal na Katulong**: Panatilihin ang konteksto sa loob ng mga araw o linggo
- **Pangangalagang Pangkalusugan**: Subaybayan ang impormasyon at kagustuhan ng pasyente
- **E-commerce**: Personal na pamimili batay sa kasaysayan

### Mga Susunod na Hakbang
- Palitan ang in-memory dict ng database o vector store (hal. Azure AI Search)
- Magdagdag ng expiration ng memorya para sa impormasyon na sensitibo sa oras
- Bumuo ng multi-agent systems na may pinagbahaging memorya
- Tuklasin ang Cognee notebook para sa knowledge-graph-backed memory


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Paalala**:  
Ang dokumentong ito ay isinalin gamit ang AI translation service na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagamat aming sinisikap ang katumpakan, mangyaring tandaan na ang mga awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o diwasto. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasaling-tao. Hindi kami mananagot sa anumang pagkakaintindihan o maling interpretasyon na maaaring magmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
